In [0]:

from pyspark.sql import functions as f 
from pyspark.sql.types import *




In [0]:
# Read orders from Bronze. Apply transformations: drop null order_id, cast total_amount to Double, cast order_date to Date, standardise status to lowercase, add processing_date
df_bronze_orders=spark.read.format('delta').table('freshcart_team1.bronze_orders')

df_silver_orders_1 = df_bronze_orders.filter(
    f.col('order_id').isNotNull()
).withColumn(
    'total_amount', f.col('total_amount').cast(DoubleType())
).withColumn(
    'order_date', f.col('order_date').cast(DateType())
).withColumn(
    'status', f.lower(f.col('status'))
).withColumn(
    'processing_date', f.current_date()
)

df_silver_orders_1.show(5)


df_silver_orders_1=df_silver_orders_1.dropDuplicates(['order_id'])



df_silver_orders_1.write.format('delta').mode('append').option("mergeSchhema",True).saveAsTable(
    'freshcart_team1.silver_orders'
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "freshcart_team1.silver_orders")

delta_table.alias("target").merge(
    df_silver_orders_1.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:

df_bronze_customers=spark.read.format('delta').table('freshcart_team1.bronze_customers')

df_bronze_customers.show(5)

row_count_before_cleaning=df_bronze_customers.count()






In [0]:
df_silver_customers_1=df_bronze_customers.dropDuplicates(['customer_id'])
df_silver_customers=df_silver_customers_1.filter(
    f.col('customer_id').isNotNull()
).withColumn(
    'city',f.initcap(f.col('city'))
)

df_silver_customers.show(5)

rows_after_clean=df_silver_customers.count()

In [0]:

print('dif in before and after cleaning is ', row_count_before_cleaning-rows_after_clean)

In [0]:

df_silver_customers.write.format('delta').mode('append').option('mergeShema',True).saveAsTable(
    'freshcart_team1.silver_customers'
)



In [0]:
from delta.tables import DeltaTable

dt=DeltaTable.forName(spark,'freshcart_team1.silver_customers')

dt.alias('target').merge(
    df_silver_customers.alias('source'),
    'target.customer_id=source.customer_id'
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
%sql

ALTER TABLE freshcart_team1.silver_orders SET TBLPROPERTIES (delta.enableChangeDataFeed = true)


In [0]:
%sql
-- # %sql
-- # S3a. Running revenue total — Using a CTE named monthly_revenue, compute
-- #      total_amount per month per city. Then use SUM() OVER (PARTITION BY city
-- #      ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
-- #      to compute a running total per city.
-- #      Show: city, month, monthly_revenue, running_total



with  monthly_revenue as (

  select city , date_format(order_date,'yyyy-MM') as month, sum(total_amount) as revenue
  from freshcart_team1.silver_orders
  group by city , date_format(order_date,'yyyy-MM')
)

select * , sum(revenue) over(PARTITION BY city
     ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
     from monthly_revenue


In [0]:
%sql
-- S3b. Customer order frequency ranking — Using a CTE named order_counts,
--      count orders per customer. Then RANK() customers within each city
--      by order count descending. Show top 3 customers per city.
--      Show: city, customer_id, order_count, city_rank
--      Filter: WHERE city_rank <= 3


with order_counts as (

  select customer_id,city, count(*) as count_order
  from freshcart_team1.silver_orders
  group by customer_id,city
)


select *  from (

select *, rank() over(partition by city order by count_order desc) as city_rank
from order_counts

)
where city_rank<=3;

